# 06: Spot Price Forecasting with LEAR (Lasso)

## Learning Goals | 学习目标

1. Understand **LEAR (Lasso Estimated AutoRegressive)** core idea / 理解 LEAR 核心思想
2. Use price lag features + L1 regularization for day-ahead prediction
3. Understand how L1 regularization auto-selects important features
4. Compare LEAR vs Persistence for price forecasting
5. Interpret Lasso coefficients (positive/negative/zero meaning)

## Data: Shandong Real-Time Price | 数据: 山东实时电价

We use the **rt_price** column from Shandong 15-min data.
Note: da_price is ~75% null, so rt_price is our primary price signal.
We filter out null values before training.

使用山东 15min 数据的 rt_price 列。da_price 约 75% 为空，rt_price 是主要价格信号。
训练前过滤掉 null 值。

---
## LEAR Principle | LEAR 原理

### Analogy: A Clever Trader's Learning Process | 类比: 精明交易员的学习过程

Imagine you are a new power trader:

**Day 1:** "Yesterday's price was 350 RMB, today should be similar" -> Persistence

**Day 2:** "Weekend prices are lower. Yesterday was Friday at 350, but today is Saturday"
-> Learning patterns

**Day 30:** "I've summarized 14 rules, but some rules are useless" -> Overconfidence

**LEAR's approach:**
- Give you ALL possible rules (14 features)
- Use L1 regularization to automatically zero out unimportant ones
- Keep only the truly predictive features

### Lasso (L1 Regularization) | Lasso 原理

**Ordinary Linear Regression:**
$$\text{Minimize: } \frac{1}{n}\sum(y_i - \hat{y}_i)^2$$

**Lasso Regression:**
$$\text{Minimize: } \frac{1}{n}\sum(y_i - \hat{y}_i)^2 + \alpha \sum|\beta_i|$$

The L1 penalty $\alpha \sum|\beta_i|$ forces unimportant feature coefficients toward zero.

| Method | Principle | Feature Selection | Interpretability |
|--------|-----------|-------------------|------------------|
| **Persistence** | yesterday = today | None | Highest |
| **LEAR (Lasso)** | Linear + L1 reg | Auto-select | High (coeff = marginal effect) |
| **XGBoost** | 100+ trees | Built-in importance | Low (black box) |

**Why LEAR works well for electricity prices?**
Prices have strong **autocorrelation** (today's price -> tomorrow's price).
A linear model with lag features captures most of the pattern.

In [ ]:
# === Imports | 导入依赖 ===
from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.price_forecaster import LEARForecaster

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'

---
## Step 1: Load & Prepare Data | 加载并准备数据

Load Shandong data, clean it, and prepare columns for LEAR.
LEARForecaster expects `price_da` as target column.
We rename `rt_price` -> `price_da` for compatibility.

LEARForecaster 需要 `price_da` 作为目标列。将 `rt_price` 重命名为 `price_da`。

In [ ]:
# Load Shandong data | 加载山东数据
loader = create_loader("shandong")
df = loader.load_data("2024-06-01", "2024-09-30")
df = clean_data(df)

print(f"Raw data: {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print()

# -- Check price columns | 检查价格列 --
print(f"rt_price nulls: {df['rt_price'].isna().sum()} / {len(df)}")
print(f"da_price nulls: {df['da_price'].isna().sum()} / {len(df)}")
print(f"rt_price range: [{df['rt_price'].min():.0f}, {df['rt_price'].max():.0f}] RMB/MWh")

# -- Prepare columns for LEAR | 为 LEAR 准备列名 --
# LEARForecaster expects: price_da, (optionally load_mw, wind_mw, solar_mw)
df['price_da'] = df['rt_price']  # use real-time price as target

# Rename wind/solar columns for LEAR Tier 2 compatibility
if 'wind_actual_mw' in df.columns:
    df['wind_mw'] = df['wind_actual_mw']
if 'solar_actual_mw' in df.columns:
    df['solar_mw'] = df['solar_actual_mw']

# -- Drop rows where rt_price is null | 去掉 rt_price 为 null 的行 --
n_before = len(df)
df = df.dropna(subset=['price_da'])
n_after = len(df)
print(f"\nDropped {n_before - n_after} rows with null price ({((n_before-n_after)/n_before*100):.1f}%)")
print(f"Clean data: {len(df):,} rows ({len(df)/96:.0f} days)")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df[['timestamp', 'load_mw', 'price_da', 'wind_mw', 'solar_mw']].head()

---
## Step 2: Feature Engineering | 特征工程

LEAR features are organized in 3 progressive tiers:

| Tier | Features | Description |
|------|----------|-------------|
| **Tier 1** | 6 | Calendar + price lags (yesterday, 7 days ago) |
| **Tier 2** | 11 (+5) | Related variable lags (load, wind, solar) + rolling stats |
| **Tier 3** | 14 (+3) | Cyclic time encoding + price trend |

Each tier builds on the previous one. Start simple, then add complexity.
For 15-min data, '24h lag' = 96 points, '168h lag' = 672 points.

In [ ]:
# === Tier 1: Calendar + Price Lags | 日历 + 价格滞后 ===
forecaster = LEARForecaster(alpha=0.01)
df = forecaster.add_price_features(df, 'tier1')

tier1_cols = forecaster.get_feature_columns('tier1')
tier1_desc = {
    'hour': 'Hour (0-23)',
    'day_of_week': 'Day of week (0=Mon, 6=Sun)',
    'month': 'Month (1-12)',
    'is_weekend': 'Is weekend (0=workday, 1=weekend)',
    'lag_24h_price': 'Price 96pts ago (yesterday same time)',
    'lag_168h_price': 'Price 672pts ago (7 days ago same time)',
}

print(f"Tier 1 features ({len(tier1_cols)}):")
for c in tier1_cols:
    print(f"  - {c:22s} {tier1_desc.get(c, '')}")
print(f"\nData shape: {df.shape}")

In [ ]:
# === Tier 2: Related Variables + Rolling Stats | 相关变量 + 滚动统计 ===
df = forecaster.add_price_features(df, 'tier2')

tier2_all = forecaster.get_feature_columns('tier2')
tier2_new = [c for c in tier2_all if c not in tier1_cols]

tier2_desc = {
    'lag_24h_load': 'Load 96pts ago (yesterday same time)',
    'lag_24h_wind': 'Wind output 96pts ago',
    'lag_24h_solar': 'Solar output 96pts ago',
    'rolling_mean_24h_price': 'Past 96pt rolling mean price',
    'rolling_std_24h_price': 'Past 96pt rolling std price',
}

print(f"Tier 2 new features ({len(tier2_new)}):")
for c in tier2_new:
    print(f"  - {c:30s} {tier2_desc.get(c, '')}")
print(f"\nTotal features: {len(tier2_all)}")

In [ ]:
# === Tier 3: Cyclic Encoding + Price Trend | 循环编码 + 价格趋势 ===
df = forecaster.add_price_features(df, 'tier3')

tier3_all = forecaster.get_feature_columns('tier3')
tier3_new = [c for c in tier3_all if c not in tier2_all]

tier3_desc = {
    'hour_sin': 'Hour sine encoding sin(2pi*h/24)',
    'hour_cos': 'Hour cosine encoding cos(2pi*h/24)',
    'price_trend_7d': '7-day moving average price',
}

print(f"Tier 3 new features ({len(tier3_new)}):")
for c in tier3_new:
    print(f"  - {c:22s} {tier3_desc.get(c, '')}")
print(f"\nTotal features: {len(tier3_all)}")
print(f"All features: {tier3_all}")

---
## Step 3: LEAR Training & Evaluation | LEAR 训练与评估

We use **TimeSeriesSplit** (n_splits=5, gap=96 for 15-min data) to prevent look-ahead bias.

### Evaluation Metrics | 评估指标

- **MAE**: Mean Absolute Error -- "on average, we're off by X RMB/MWh"
- **RMSE**: Root Mean Squared Error -- penalizes large errors more heavily
- **MAPE**: Mean Absolute Percentage Error -- error as % of actual price
- **R^2**: Coefficient of Determination -- 1=perfect, 0=no better than mean

### Baseline: Persistence | 基线: 持续法

Persistence assumes **today's price = yesterday's same-moment price**.
If LEAR can't beat persistence, something is wrong with features or parameters.

In [ ]:
# === Train LEAR Model (Tier 3) | 训练 LEAR 模型 ===
n_splits = 5
gap = 96  # 1 day gap for 15-min data

result = forecaster.train_evaluate(df, tier='tier3', n_splits=n_splits, gap=gap)

metrics = result['metrics']
predictions = result['predictions']
actuals = result['actuals']

# Compute supplementary metrics
mape_val = np.mean(np.abs((actuals - predictions) / np.where(actuals != 0, actuals, 1e-10))) * 100
r2 = r2_score(actuals, predictions)

print("=" * 50)
print("LEAR (Tier 3) Evaluation Results")
print("=" * 50)
print(f"MAE:  {metrics['mae']:.2f} RMB/MWh")
print(f"RMSE: {metrics['rmse']:.2f} RMB/MWh")
print(f"MAPE: {mape_val:.1f}%")
print(f"R^2:  {r2:.3f}")

# Feature coefficient stats
coefs = result['model'].coef_
n_zero = sum(1 for c in coefs if abs(c) < 1e-10)
print(f"\nLasso coefficients: {n_zero}/{len(coefs)} zeroed out ({n_zero/len(coefs)*100:.0f}% sparsity)")

# Compare with persistence baseline
persist_vals = df['price_da'].shift(96)
valid_mask = persist_vals.notna()
persist_mae = (persist_vals[valid_mask] - df.loc[valid_mask, 'price_da']).abs().mean()

print(f"\n{'='*50}")
print("Model Comparison")
print(f"{'='*50}")
print(f"Persistence MAE: {persist_mae:.2f} RMB/MWh")
print(f"LEAR MAE:        {metrics['mae']:.2f} RMB/MWh")

improvement = (persist_mae - metrics['mae']) / persist_mae * 100 if persist_mae > 0 else 0
if improvement > 0:
    print(f"\nLEAR improves by {improvement:.1f}% -- better than persistence!")
else:
    print(f"\nLEAR underperforms persistence by {abs(improvement):.1f}%. Check features or alpha.")

---
## Step 4: Visualization | 可视化

Two plots:
1. **Actual vs Predicted overlay** -- time series comparison
2. **Error distribution histogram** -- is error centered at zero?

In [ ]:
# === Plot 1: LEAR Forecast vs Actual | 预测 vs 实际 ===
fig_forecast = forecaster.plot_price_forecast(df, predictions,
    title="LEAR Day-Ahead Price Forecast vs Actual (Shandong 15-min)")
fig_forecast.show()

# Print error statistics
errors = actuals - predictions
print(f"Error mean:     {errors.mean():.2f} RMB/MWh")
print(f"Error median:   {np.median(errors):.2f} RMB/MWh")
print(f"Error std:      {errors.std():.2f} RMB/MWh")
print(f"Within +/-1 MAE: {(np.abs(errors) <= metrics['mae']).mean()*100:.1f}%")

In [ ]:
# === Plot 2: Lasso Feature Coefficients | Lasso 特征系数 ===
try:
    fig_coef = forecaster.plot_coefficients(top_n=14,
        title="LEAR Feature Coefficients (L1 Regularization, Shandong 15-min)")
    fig_coef.show()
except Exception as e:
    print(f"Could not plot coefficients: {e}")

# Print coefficient ranking
coef_df = pd.DataFrame({
    'Feature': forecaster.get_feature_columns('tier3'),
    'Coefficient': result['model'].coef_,
    '|Coeff|': np.abs(result['model'].coef_),
}).sort_values('|Coeff|', ascending=False)

print("\nFeature Coefficient Ranking (by |coeff|):")
print(f"{'Feature':<25s} {'Coeff':>10s}  Interpretation")
print("-" * 65)
for _, row in coef_df.iterrows():
    bar = chr(9608) * int(abs(row['Coefficient']) / coef_df['|Coeff|'].max() * 20)
    if abs(row['Coefficient']) < 1e-10:
        direction = '(removed by L1)'
    elif row['Coefficient'] > 0:
        direction = 'pushes price UP'
    else:
        direction = 'pushes price DOWN'
    print(f"  {row['Feature']:<23s} {row['Coefficient']:+10.4f}  {bar} {direction}")

---
## Step 5: Error Analysis by Time | 按时间维度分析误差

Does LEAR perform better at certain hours? Check error by hour of day.

In [ ]:
# Error by hour | 按小时分析误差
n_pred = len(predictions)
error_df = pd.DataFrame({
    'hour': df['timestamp'].dt.hour.values[-n_pred:],
    'abs_error': np.abs(errors),
})
hourly_mae = error_df.groupby('hour')['abs_error'].mean()

fig_hourly = go.Figure()
fig_hourly.add_trace(go.Bar(
    x=list(range(24)), y=hourly_mae.values,
    marker_color=['#d62728' if v == hourly_mae.max() else '#1f77b4' for v in hourly_mae.values],
    text=[f'{v:.1f}' for v in hourly_mae.values],
    textposition='outside',
))
fig_hourly.update_layout(
    title="LEAR MAE by Hour of Day (Shandong 15-min)",
    xaxis_title="Hour", yaxis_title="MAE (RMB/MWh)",
    height=400, showlegend=False
)
fig_hourly.show()

best_h = hourly_mae.idxmin()
worst_h = hourly_mae.idxmax()
print(f"Best hour:  {best_h}:00 (MAE={hourly_mae.min():.1f})")
print(f"Worst hour: {worst_h}:00 (MAE={hourly_mae.max():.1f})")
print()
print("Why are certain hours harder? Peak/ramp hours have higher price volatility.")

---
## Key Takeaways | 学习笔记

- **LEAR** = Lasso + lag features, a classic baseline for price forecasting
- **L1 regularization** auto-selects features -- unimportant ones get zeroed
- **Positive coefficient** = feature up -> price up; **Negative** = feature up -> price down
- **TimeSeriesSplit(gap=96)** prevents look-ahead bias on 15-min data
- **Persistence is the minimum bar** -- any model should beat it
- **15-min data** reveals intra-day price patterns invisible in hourly data

## Reflection Questions | 反思题

1. Which features got zeroed by L1? Why might they be irrelevant?
2. If persistence beats LEAR, what could be wrong? List 3 possible reasons.
3. How does cyclic encoding (hour_sin/hour_cos) help compared to raw hour (0-23)?
4. Why do certain hours (e.g., ramp periods) have higher MAE?
5. How would you adapt this model for real trading decisions?

Next: Notebook 07 -> Multi-Model Comparison Dashboard.